# ECHO — a spatial delay-line instrument (pyquist)

Place sounds in a virtual room, move them along paths, and hear **interaural time differences, Doppler shift, distance cues, room reflections, a feedback-delay-network reverb and a ping-pong delay** — all built from delay lines implemented in NumPy (`echo/dsp.py`, `echo/spatial.py`).

Run the cells top to bottom, **use headphones**. The DSP lives in the `echo/` package next to this notebook; this notebook is just the control panel (see `TECHNICAL.md` for every parameter).

```
pip install pyquist matplotlib
```

In [1]:
import numpy as np
import pyquist as pq
import matplotlib.pyplot as plt

from echo import (Echo, Room, Propagation, ReverbSettings, DelaySettings,
                  Static, Orbit, Spiral, Oscillate, Flyby, Waypoints, sources, dsp)

## 1. Interactive instrument
Pick a demo or add sounds, drag them on the map, shape the room with the sliders, and press **Render and play**. The panel opens below; to use the whole window run `python app.py` in a terminal instead.

In [2]:
from echo.webui import serve_background
from IPython.display import IFrame

IFrame(serve_background(), width="100%", height=780)

## 2. Same thing from code (what the demo video uses)
The engine is also a plain Python library — useful inside the final composition.

In [ ]:
e = Echo()
e.room = Room(width=14, depth=11, absorption=0.25)
e.reverb = ReverbSettings(kind="fdn", t60=3.0, damping=0.25, level_db=-8)
e.delay = DelaySettings(enabled=True, time=0.375, feedback=0.5, level_db=-9)

arp = sources.pluck_line([(i * 0.25, p, 2.0) for i, p in enumerate(["A3","C4","D4","E4","G4","A4","E4","D4"] * 2)],
                         t60=2.0, damping=0.4, pick_position=0.2)
e.add_source(arp, "arp", motion=Orbit(radius=3.5, speed=0.08), send=0.35)

out = e.render(8.0)
pq.play(out, normalize=True)
e.plot_scene(seconds=8.0);

## 3. Looking inside: the building blocks
The block-vectorized feedback delay reproduces the circular-buffer loop from the notes exactly:

In [ ]:
def feedback_delay_notes(x, M, g):          # pcm-13: circular buffer, one sample at a time
    buf, i, y = np.zeros(M), 0, np.zeros(len(x))
    for n in range(len(x)):
        y[n] = x[n] + g * buf[i]; buf[i] = y[n]; i = (i + 1) % M
    return y

x = np.random.randn(4000)
print(np.allclose(feedback_delay_notes(x, 441, 0.8), dsp.feedback_delay(x, 441, 0.8)))

Doppler falls out of the fractional delay line — a source flying past at 15 m/s:

In [ ]:
fs = 44100
t = np.arange(4 * fs) / fs
px, py = Flyby(-30, 2, 30, 2, duration=4.0)(t)
from echo.spatial import propagate
y = propagate(np.sin(2*np.pi*1000*t), px, py, fs, Propagation())
pq.plot_spec(pq.Audio(y.astype(np.float32), fs)) if hasattr(pq, "plot_spec") else None
pq.play(pq.Audio(y.astype(np.float32), fs), normalize=True)